In [1]:
import os
import json
import yaml
from src.utils.file_utils import find_config_with_conditions
from typing import Iterable, List, Union

In [2]:
def match_conditions(config, conditions):
    """
    递归匹配 config 是否满足 conditions 条件
    """
    for k, v in conditions.items():
        if k not in config:
            return False
        if isinstance(v, dict):
            if not isinstance(config[k], dict):
                return False
            if not match_conditions(config[k], v):
                return False
        else:
            if config[k] != v:
                return False
    return True


def _load_config(config_path: str):
    """
    根据扩展名读取配置，支持 .json / .yaml / .yml
    """
    ext = os.path.splitext(config_path)[1].lower()
    with open(config_path, "r", encoding="utf-8") as f:
        if ext == ".json":
            return json.load(f)
        elif ext in (".yaml", ".yml"):
            return yaml.safe_load(f)
        else:
            raise ValueError(f"不支持的配置格式: {ext}")


def find_config_with_conditions(
    conditions: dict,
    root_dir: str,
    filenames: Union[str, Iterable[str]] = ("config.yaml", "config.json"),
) -> List[str]:
    """
    根据给定的条件递归筛选配置文件，并返回所有符合条件的文件夹绝对路径。

    - filenames: 可以是单个文件名字符串（如 "config.yaml"），
                 也可以是多个文件名（如 ("config.yaml","config.json")）。
                 默认同时支持 yaml/yml/json。
    """
    # 统一成 set，避免重复 & 加速查询
    if isinstance(filenames, str):
        target_names = {filenames}
    else:
        target_names = set(filenames)

    matching_dirs = []

    for subdir, _, files in os.walk(root_dir):
        # 只在当前目录中存在目标文件名之一时才尝试读取
        hit_names = target_names.intersection(files)
        for name in hit_names:
            config_path = os.path.join(subdir, name)
            try:
                config = _load_config(config_path)
                if isinstance(config, dict) and match_conditions(config, conditions):
                    matching_dirs.append(os.path.abspath(subdir))
                    # 命中一个就可加入；若同目录多个文件名都想验证，可不 break
                    break
            except Exception as e:
                print(f"[错误] 读取 {config_path} 时失败: {e}")

    return matching_dirs


In [3]:
root_dir = './checkpoints'
condition1 = {
    # 'load_specific_parts': ['encoder'],
    'model': 'clf_mixer_attnpl_t',
    'mixer_model_config': {
        # 'd_model': 64,
        # 'attn_layer_idx': [],
        # 'ssm_cfg':{'layer': 'Mamba2'}
    },
    'dataset':'QTMSaltonSea',
    # 'Mf': 4.5,
    # 'Twindow': 180,
    # 'Tfore': 20
}
condition2 = {
    'model': 'mixer_tpp',
    'mixer_model_config': {
    #     # 'd_model': 64,
        # 'attn_layer_idx': [0],
        # 'ssm_cfg':{'layer': 'Mamba2'}
    },
    # 'predict_b':  False,
    'features_input_keys': ['mag', ],  # 'log_inter_times'
    'dataset':'ChuanDian'
}

condition3 = {
    'model': 'rtpp',
    # 'mixer_model_config': {
    #     # 'd_model': 64,
    #     # 'attn_layer_idx': [],
    #     # 'ssm_cfg':{'layer': 'Mamba2'}
    # },
    # "dMag": 0.1,
    'dataset':'ChuanDian'
}

condition4 = {
    'model': 'lstm',
    'dataset':'ChuanDian'
}

condition5 = {
    'model': 'reg_mixer_attnpl_t',
    'dataset':'ChuanDian',
    'Twindow': 600,
}
condition6 = {
    # 'model': 'rf',
  "Mc": 0.6,
  "Mf": 3.7,
  "Twindow": 180,
  "Tfore": 60,
  "dt": 5,
  "context_len": 1,
    # 'dataset':'ChuanDian',
    # "Mag_elaps": "[5, 5.5, 6, 6.5]"
}
dirs = find_config_with_conditions(condition2, root_dir)

In [4]:
dirs 
# dirs = ["checkpoints/rf_847e64da",
#         "checkpoints/rf_dfae9b13",
#         "checkpoints/rf_ba2359e6",
#         "checkpoints/rf_9ffe46be",
#         "checkpoints/rf_af684ff4"]

['/root/autodl-tmp/chuandian_eq/checkpoints/mixer_tpp_20250912-202304',
 '/root/autodl-tmp/chuandian_eq/checkpoints/mixer_tpp_20250816-130213',
 '/root/autodl-tmp/chuandian_eq/checkpoints/mixer_tpp_20250816-131206',
 '/root/autodl-tmp/chuandian_eq/checkpoints/mixer_tpp_20250816-210231',
 '/root/autodl-tmp/chuandian_eq/checkpoints/mixer_tpp_20250817-163813',
 '/root/autodl-tmp/chuandian_eq/checkpoints/mixer_tpp_20250817-164710',
 '/root/autodl-tmp/chuandian_eq/checkpoints/mixer_tpp_20250817-173915',
 '/root/autodl-tmp/chuandian_eq/checkpoints/mixer_tpp_20250817-174809',
 '/root/autodl-tmp/chuandian_eq/checkpoints/mixer_tpp_20250817-175122',
 '/root/autodl-tmp/chuandian_eq/checkpoints/mixer_tpp_20250817-175515',
 '/root/autodl-tmp/chuandian_eq/checkpoints/mixer_tpp_20250817-220423',
 '/root/autodl-tmp/chuandian_eq/checkpoints/mixer_tpp_20250817-222803',
 '/root/autodl-tmp/chuandian_eq/checkpoints/mixer_tpp_20250818-103012',
 '/root/autodl-tmp/chuandian_eq/checkpoints/mixer_tpp_20250818-1

查找checkpoint

In [9]:
def get_metrics_json_from_file_list(file_list, name):
    """
    根据给定的文件列表，检查每个文件夹是否包含 metrics.json 文件，并返回其内容
    
    :param file_list: 包含文件夹路径的列表
    :return: 返回一个包含文件夹路径和对应的 metrics.json 数据的字典列表
    """
    result = [] 

    # 遍历 file_list 中的每个文件夹路径
    for folder_path in file_list:
        metrics_path = os.path.join(folder_path, name)  # 拼接 metrics.json 的路径
        if os.path.isfile(metrics_path):  # 如果 metrics.json 文件存在
            try:
                # 读取并解析 metrics.json 文件
                with open(metrics_path, 'r') as metrics_file:
                    metrics_data = json.load(metrics_file)
                
                # 将文件夹路径和对应的 metrics.json 数据添加到结果列表中
                result.append({
                    'folder_path': os.path.abspath(folder_path),
                    'metrics_data': metrics_data
                })
            except Exception as e:
                print(f"读取 {metrics_path} 文件时发生错误: {e}")
        else:
            print(f"文件夹 {folder_path} 中未找到 metrics.json 文件")

    # 返回符合条件的所有文件夹路径和对应的 metrics.json 数据
    return result


json_files = get_metrics_json_from_file_list(dirs, 'metrics_test_best.json')

# 打印结果
if json_files:
    print("找到的文件夹和对应的 metrics.json 内容:")
    for item in json_files:
        print(f"文件夹路径: {item['folder_path']}")
        print(f"metrics.json 内容: {item['metrics_data']}")
else:
    print("没有找到包含 metrics.json 文件的目录")


文件夹 /root/autodl-tmp/chuandian_eq/checkpoints/mixer_tpp_20250816-130213 中未找到 metrics.json 文件
文件夹 /root/autodl-tmp/chuandian_eq/checkpoints/mixer_tpp_20250816-131206 中未找到 metrics.json 文件
文件夹 /root/autodl-tmp/chuandian_eq/checkpoints/mixer_tpp_20250816-210231 中未找到 metrics.json 文件
文件夹 /root/autodl-tmp/chuandian_eq/checkpoints/mixer_tpp_20250817-163813 中未找到 metrics.json 文件
文件夹 /root/autodl-tmp/chuandian_eq/checkpoints/mixer_tpp_20250817-164710 中未找到 metrics.json 文件
文件夹 /root/autodl-tmp/chuandian_eq/checkpoints/mixer_tpp_20250817-173915 中未找到 metrics.json 文件
文件夹 /root/autodl-tmp/chuandian_eq/checkpoints/mixer_tpp_20250817-174809 中未找到 metrics.json 文件
文件夹 /root/autodl-tmp/chuandian_eq/checkpoints/mixer_tpp_20250817-175122 中未找到 metrics.json 文件
文件夹 /root/autodl-tmp/chuandian_eq/checkpoints/mixer_tpp_20250817-175515 中未找到 metrics.json 文件
文件夹 /root/autodl-tmp/chuandian_eq/checkpoints/mixer_tpp_20250817-220423 中未找到 metrics.json 文件
文件夹 /root/autodl-tmp/chuandian_eq/checkpoints/mixer_tpp_20250817-22280

In [31]:
target = 0.587

result = [item for item in json_files 
          if abs(float(item['metrics_data']['nll_test_time']) - target) < 1e-3]

print(result)


[{'folder_path': '/root/autodl-tmp/chuandian_eq/checkpoints/mixer_tpp_20250909-114545', 'metrics_data': {'nll_train_time': 0.3149864971637726, 'nll_train_mag': 0.35133710503578186, 'nll_train_total': 0.6575055718421936, 'nll_train_b': -0.8818033933639526, 'nll_val_time': 0.817458987236023, 'nll_val_mag': 0.0935339629650116, 'nll_val_total': 0.9076213836669922, 'nll_val_b': -0.33715760707855225, 'nll_test_time': 0.5872343182563782, 'nll_test_mag': 0.26162248849868774, 'nll_test_total': 0.8420678377151489, 'nll_test_b': -0.678908109664917, 'num_events_train': 4498, 'num_events_val': 567}}]


In [28]:
test_json_files = get_metrics_json_from_file_list(dirs, 'test_metrics.json')
val_json_files = get_metrics_json_from_file_list(dirs, 'val_metrics.json')

In [40]:
test_json_files

[{'folder_path': '/home/yzzhang/pjt/chuandian_eq/checkpoints/rf_847e64da',
  'metrics_data': {'precision': 0.8,
   'recall': 0.45714285714285713,
   'f1': 0.5818181818181818,
   'auc': 0.8212389380530973,
   'fpr': 0.035398230088495575,
   'tpr': 0.45714285714285713,
   'R': 0.3657142857142857,
   'conf': 1.0,
   'threshold': 0.6364525402815664}},
 {'folder_path': '/home/yzzhang/pjt/chuandian_eq/checkpoints/rf_dfae9b13',
  'metrics_data': {'precision': 0.5639097744360902,
   'recall': 0.9493670886075949,
   'f1': 0.7075471698113207,
   'auc': 0.6829858525688757,
   'fpr': 0.8529411764705882,
   'tpr': 0.9493670886075949,
   'R': 0.5353573807937565,
   'conf': 1.9024795333968695e-16,
   'threshold': 0.5646362918030292}},
 {'folder_path': '/home/yzzhang/pjt/chuandian_eq/checkpoints/rf_ba2359e6',
  'metrics_data': {'precision': 0.5833333333333334,
   'recall': 0.8936170212765957,
   'f1': 0.7058823529411765,
   'auc': 0.871343085106383,
   'fpr': 0.3125,
   'tpr': 0.8936170212765957,
   '

In [54]:
import pandas as pd

data = []
for item in val_json_files:
    folder_path = item['folder_path']
    metrics_data = item['metrics_data']
    row = {'folder_path': folder_path}
    row.update(metrics_data)
    data.append(row)

df = pd.DataFrame(data)
# df = df.sort_values(by='folder_path').reset_index(drop=True)

# 只保留指定列
df = df[['folder_path', 'auc', 'f1', 'recall', 'precision', 'R']]

print(df)


                                         folder_path       auc        f1  \
0  /home/yzzhang/pjt/chuandian_eq/checkpoints/rf_...  0.622093  0.615385   
1  /home/yzzhang/pjt/chuandian_eq/checkpoints/rf_...  0.611191  0.849206   
2  /home/yzzhang/pjt/chuandian_eq/checkpoints/rf_...  0.706827  0.764977   
3  /home/yzzhang/pjt/chuandian_eq/checkpoints/rf_...  0.879409  0.842697   
4  /home/yzzhang/pjt/chuandian_eq/checkpoints/rf_...  0.859767  0.765957   

     recall  precision         R  
0  1.000000   0.444444  0.127907  
1  1.000000   0.737931  0.025641  
2  1.000000   0.619403  0.150000  
3  0.862069   0.824176  0.586207  
4  0.870968   0.683544  0.593190  


In [53]:
df['folder_path'][3]

'/home/yzzhang/pjt/chuandian_eq/checkpoints/rf_9ffe46be'

按日期删除checkpoint

In [7]:
import os
import shutil
from datetime import datetime

# 设置目标目录
checkpoint_dir = "checkpoints"
# 指定保留起点（目标时间），格式必须和文件夹一致
threshold = "20250518-205722"
threshold_dt = datetime.strptime(threshold, "%Y%m%d-%H%M%S")

# 遍历文件夹
for folder in os.listdir(checkpoint_dir):
    folder_path = os.path.join(checkpoint_dir, folder)
    
    if os.path.isdir(folder_path) and folder.startswith("classifier_"):
        time_str = folder.split("_")[1]
        folder_dt = datetime.strptime(time_str, "%Y%m%d-%H%M%S")

        if folder_dt < threshold_dt:
            print(f"删除：{folder_path}")
            shutil.rmtree(folder_path)  # 删除整个文件夹


ValueError: time data 'se' does not match format '%Y%m%d-%H%M%S'